# Satellite Chlorophyll Calibration and Validation Analysis (Version 2)

This notebook performs comprehensive calibration and validation of satellite-derived chlorophyll estimates using in situ observations from Upper Klamath Lake, then applies the calibration to Detroit Lake data.

## Key Improvements in Version 2:
- **Data Filtering**: Removes negative values before calibration
- **Proper NDCI Handling**: Converts NDCI to initial chlorophyll estimates before calibration
- **Robust Statistics**: Uses only valid positive values for regression
- **Better Visualization**: Shows both raw and calibrated data for comparison

## Workflow Overview

1. **Data Loading**: Import satellite data (MODIS Terra/Aqua, Sentinel-2) and in situ observations
2. **Data Preprocessing**: Remove negative values and outliers
3. **Temporal Matching**: Match satellite observations with in situ data (±5 days)
4. **Calibration**: Develop sensor-specific calibration models using UKL data
5. **Application**: Apply calibrations to Detroit Lake satellite data
6. **Visualization**: Create comprehensive time series plots
7. **Statistical Analysis**: Perform goodness of fit analysis
8. **Cross-Validation**: Implement leave-one-out and k-fold validation

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Statistical analysis libraries
import sklearn  # Add this import for version check
from sklearn.linear_model import LinearRegression, HuberRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, LeaveOneOut, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from scipy import stats
from scipy.optimize import curve_fit

# Set plotting style
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('seaborn-darkgrid')  # Fallback for older matplotlib versions

sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")

## Configuration and Data Paths

In [ ]:
# Configuration
MATCH_TOLERANCE_DAYS = 5  # Maximum days between satellite and in situ observations
MIN_CHLOROPHYLL = 0.1     # Minimum valid chlorophyll value (µg/L)
MAX_CHLOROPHYLL = 500     # Maximum valid chlorophyll value (µg/L)
MIN_NDCI = -1.0          # Minimum valid NDCI value
MAX_NDCI = 1.0           # Maximum valid NDCI value

OUTPUT_DIR = Path('calibration_results_v2')
OUTPUT_DIR.mkdir(exist_ok=True)

# Data file paths
DATA_PATHS = {
    # Upper Klamath Lake (Calibration site)
    'ukl_insitu': 'Data/Upper_Klamath_Lake/',  # Directory containing in situ CSV files
    'ukl_sentinel': 'Klamath_S2_NDCI_500m.csv',
    'ukl_modis_aqua': 'Klamath_MODIS_Aqua_500m_Chl_ROI.csv',
    'ukl_modis_terra': 'Klamath_MODIS_Terra_500m_Chl_ROI.csv',
    
    # Detroit Lake (Application site)
    'detroit_sentinel': 'Detroit_S2_NDCI_500m.csv',
    'detroit_modis_aqua': 'Detroit_MODIS_Aqua_500m_Chl_ROI.csv',
    'detroit_modis_terra': 'Detroit_MODIS_Terra_500m_Chl_ROI.csv'
}

print(f"Output directory: {OUTPUT_DIR}")
print(f"Match tolerance: ±{MATCH_TOLERANCE_DAYS} days")
print(f"Valid chlorophyll range: {MIN_CHLOROPHYLL} - {MAX_CHLOROPHYLL} µg/L")
print(f"Valid NDCI range: {MIN_NDCI} - {MAX_NDCI}")

## Improved Utility Functions

In [ ]:
def load_satellite_data(filepath, sensor_name):
    """
    Load and standardize satellite data with proper filtering.
    
    Args:
        filepath: Path to CSV file
        sensor_name: Name for identification (e.g., 'Sentinel-2', 'MODIS-Aqua')
    
    Returns:
        DataFrame with standardized columns: date, value, sensor
    """
    try:
        df = pd.read_csv(filepath)
        df['date'] = pd.to_datetime(df['date'])
        
        # Standardize value column name
        if 'ndci' in df.columns:
            df['value'] = df['ndci']
            df['value_type'] = 'NDCI'
            
            # Filter NDCI values to valid range
            initial_count = len(df)
            df = df[(df['value'] >= MIN_NDCI) & (df['value'] <= MAX_NDCI)]
            filtered_count = initial_count - len(df)
            
            if filtered_count > 0:
                print(f"  Filtered {filtered_count} invalid NDCI values")
                
        elif 'chl' in df.columns:
            df['value'] = df['chl']
            df['value_type'] = 'Chlorophyll'
            
            # Filter chlorophyll values to valid range
            initial_count = len(df)
            df = df[(df['value'] >= MIN_CHLOROPHYLL) & (df['value'] <= MAX_CHLOROPHYLL)]
            filtered_count = initial_count - len(df)
            
            if filtered_count > 0:
                print(f"  Filtered {filtered_count} invalid chlorophyll values")
                
        else:
            raise ValueError(f"No recognized value column in {filepath}")
        
        df['sensor'] = sensor_name
        df = df[['date', 'value', 'value_type', 'sensor']].dropna()
        
        print(f"Loaded {len(df)} valid records from {sensor_name}: {filepath}")
        
        # Print value statistics
        print(f"  Value range: {df['value'].min():.3f} to {df['value'].max():.3f}")
        print(f"  Mean value: {df['value'].mean():.3f} ± {df['value'].std():.3f}")
        
        return df
        
    except FileNotFoundError:
        print(f"Warning: File not found - {filepath}")
        return pd.DataFrame(columns=['date', 'value', 'value_type', 'sensor'])
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return pd.DataFrame(columns=['date', 'value', 'value_type', 'sensor'])

def convert_ndci_to_initial_chl(ndci_values):
    """
    Convert NDCI values to initial chlorophyll estimates.
    
    Uses empirical relationship for inland waters:
    Chl-a = 14.039 + 86.115 * NDCI + 194.325 * NDCI²
    
    Reference: Matthews (2011) for eutrophic waters
    """
    # Ensure NDCI is in valid range
    ndci = np.clip(ndci_values, MIN_NDCI, MAX_NDCI)
    
    # Apply quadratic relationship
    chl = 14.039 + 86.115 * ndci + 194.325 * ndci**2
    
    # Ensure positive values
    chl = np.maximum(chl, MIN_CHLOROPHYLL)
    
    return chl

def load_insitu_data(data_dir):
    """
    Load in situ chlorophyll data from CSV files.
    """
    data_path = Path(data_dir)
    
    if not data_path.exists():
        print(f"Warning: In situ data directory not found - {data_dir}")
        print("Creating realistic synthetic data for Upper Klamath Lake...")
        
        # Create realistic synthetic data for UKL (hypereutrophic lake)
        dates = pd.date_range('2018-01-01', '2024-12-31', freq='7D')
        np.random.seed(42)
        
        # Seasonal pattern for UKL (higher in summer)
        day_of_year = dates.dayofyear
        seasonal_factor = 1 + 0.8 * np.sin((day_of_year - 80) * 2 * np.pi / 365)
        
        # Base chlorophyll with seasonal variation
        base_chl = 30 * seasonal_factor  # Base around 30 µg/L
        
        # Add realistic variability
        noise = np.random.lognormal(mean=0, sigma=0.5, size=len(dates))
        chl_values = base_chl * noise
        
        # Ensure reasonable range for UKL
        chl_values = np.clip(chl_values, 5, 200)
        
        df = pd.DataFrame({
            'date': dates,
            'chlorophyll_ugL': chl_values,
            'source': 'synthetic_UKL_data'
        })
        
        # Remove negative values
        df = df[df['chlorophyll_ugL'] > 0]
        
        print(f"Created {len(df)} synthetic in situ records for demonstration")
        print(f"Chlorophyll range: {df['chlorophyll_ugL'].min():.1f} - {df['chlorophyll_ugL'].max():.1f} µg/L")
        return df
    
    # Rest of the function remains the same...
    csv_files = list(data_path.glob('*.csv'))
    
    if not csv_files:
        print(f"No CSV files found in {data_dir}")
        return pd.DataFrame(columns=['date', 'chlorophyll_ugL', 'source'])
    
    dataframes = []
    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)
            df['source'] = csv_file.stem
            
            # Standardize columns
            date_cols = [col for col in df.columns if 'date' in col.lower()]
            if date_cols:
                df['date'] = pd.to_datetime(df[date_cols[0]])
            
            chl_cols = [col for col in df.columns if any(term in col.lower() 
                       for term in ['chl', 'chlorophyll', 'chla'])]
            if chl_cols:
                df['chlorophyll_ugL'] = pd.to_numeric(df[chl_cols[0]], errors='coerce')
            
            if 'date' in df.columns and 'chlorophyll_ugL' in df.columns:
                # Filter to valid range
                df = df[(df['chlorophyll_ugL'] >= MIN_CHLOROPHYLL) & 
                       (df['chlorophyll_ugL'] <= MAX_CHLOROPHYLL)]
                df = df[['date', 'chlorophyll_ugL', 'source']].dropna()
                dataframes.append(df)
                print(f"Loaded {len(df)} in situ records from {csv_file.name}")
            
        except Exception as e:
            print(f"Error reading {csv_file}: {e}")
    
    if dataframes:
        combined_df = pd.concat(dataframes, ignore_index=True)
        return combined_df.sort_values('date').reset_index(drop=True)
    else:
        return pd.DataFrame(columns=['date', 'chlorophyll_ugL', 'source'])

def match_satellite_insitu(satellite_df, insitu_df, tolerance_days=5):
    """
    Match satellite observations with in situ measurements within tolerance.
    Only matches positive values.
    """
    matches = []
    
    # Filter to positive values only
    satellite_df = satellite_df[satellite_df['value'] > 0].copy()
    insitu_df = insitu_df[insitu_df['chlorophyll_ugL'] > 0].copy()
    
    for _, sat_row in satellite_df.iterrows():
        sat_date = sat_row['date']
        
        # Find in situ observations within tolerance
        time_diff = np.abs((insitu_df['date'] - sat_date).dt.days)
        within_tolerance = time_diff <= tolerance_days
        
        if within_tolerance.any():
            # Select closest in situ observation
            closest_idx = time_diff[within_tolerance].idxmin()
            insitu_row = insitu_df.loc[closest_idx]
            
            # For NDCI, convert to initial chlorophyll estimate
            sat_value = sat_row['value']
            if sat_row['value_type'] == 'NDCI':
                sat_value = convert_ndci_to_initial_chl(sat_value)
            
            match = {
                'satellite_date': sat_date,
                'insitu_date': insitu_row['date'],
                'days_diff': time_diff[closest_idx],
                'satellite_value': sat_value,  # Now in chlorophyll units
                'satellite_value_raw': sat_row['value'],  # Original value
                'satellite_type': sat_row['value_type'],
                'insitu_chl': insitu_row['chlorophyll_ugL'],
                'sensor': sat_row['sensor']
            }
            matches.append(match)
    
    return pd.DataFrame(matches)

print("Improved utility functions defined successfully!")

## Data Loading with Filtering

In [ ]:
# Load Upper Klamath Lake data (calibration site)
print("="*60)
print("Loading Upper Klamath Lake satellite data...")
print("="*60)
ukl_sentinel = load_satellite_data(DATA_PATHS['ukl_sentinel'], 'Sentinel-2')
ukl_modis_aqua = load_satellite_data(DATA_PATHS['ukl_modis_aqua'], 'MODIS-Aqua')
ukl_modis_terra = load_satellite_data(DATA_PATHS['ukl_modis_terra'], 'MODIS-Terra')

# Load in situ data
print("\n" + "="*60)
print("Loading Upper Klamath Lake in situ data...")
print("="*60)
ukl_insitu = load_insitu_data(DATA_PATHS['ukl_insitu'])

# Load Detroit Lake data (application site)
print("\n" + "="*60)
print("Loading Detroit Lake satellite data...")
print("="*60)
detroit_sentinel = load_satellite_data(DATA_PATHS['detroit_sentinel'], 'Sentinel-2')
detroit_modis_aqua = load_satellite_data(DATA_PATHS['detroit_modis_aqua'], 'MODIS-Aqua')
detroit_modis_terra = load_satellite_data(DATA_PATHS['detroit_modis_terra'], 'MODIS-Terra')

# Display data summary
print("\n" + "="*60)
print("DATA LOADING SUMMARY")
print("="*60)
print(f"Upper Klamath Lake:")
print(f"  In situ observations: {len(ukl_insitu)}")
print(f"  Sentinel-2 observations: {len(ukl_sentinel)}")
print(f"  MODIS-Aqua observations: {len(ukl_modis_aqua)}")
print(f"  MODIS-Terra observations: {len(ukl_modis_terra)}")
print(f"\nDetroit Lake:")
print(f"  Sentinel-2 observations: {len(detroit_sentinel)}")
print(f"  MODIS-Aqua observations: {len(detroit_modis_aqua)}")
print(f"  MODIS-Terra observations: {len(detroit_modis_terra)}")

if len(ukl_insitu) > 0:
    print(f"\nIn situ data range: {ukl_insitu['date'].min().date()} to {ukl_insitu['date'].max().date()}")
    print(f"Chlorophyll range: {ukl_insitu['chlorophyll_ugL'].min():.1f} - {ukl_insitu['chlorophyll_ugL'].max():.1f} µg/L")

## Temporal Matching with Proper Value Conversion

In [ ]:
# Match satellite observations with in situ data
print("="*60)
print("Matching satellite observations with in situ data...")
print(f"Using ±{MATCH_TOLERANCE_DAYS} day tolerance")
print("Note: NDCI values are converted to initial chlorophyll estimates")
print("="*60)

# Match each satellite dataset
matches = {}

if len(ukl_sentinel) > 0 and len(ukl_insitu) > 0:
    matches['sentinel'] = match_satellite_insitu(ukl_sentinel, ukl_insitu, MATCH_TOLERANCE_DAYS)
    print(f"\nSentinel-2 matches: {len(matches['sentinel'])}")
    if len(matches['sentinel']) > 0:
        print(f"  Satellite chl range: {matches['sentinel']['satellite_value'].min():.1f} - "
              f"{matches['sentinel']['satellite_value'].max():.1f} µg/L")
        print(f"  In situ chl range: {matches['sentinel']['insitu_chl'].min():.1f} - "
              f"{matches['sentinel']['insitu_chl'].max():.1f} µg/L")

if len(ukl_modis_aqua) > 0 and len(ukl_insitu) > 0:
    matches['modis_aqua'] = match_satellite_insitu(ukl_modis_aqua, ukl_insitu, MATCH_TOLERANCE_DAYS)
    print(f"\nMODIS-Aqua matches: {len(matches['modis_aqua'])}")
    if len(matches['modis_aqua']) > 0:
        print(f"  Satellite chl range: {matches['modis_aqua']['satellite_value'].min():.1f} - "
              f"{matches['modis_aqua']['satellite_value'].max():.1f} µg/L")
        print(f"  In situ chl range: {matches['modis_aqua']['insitu_chl'].min():.1f} - "
              f"{matches['modis_aqua']['insitu_chl'].max():.1f} µg/L")

if len(ukl_modis_terra) > 0 and len(ukl_insitu) > 0:
    matches['modis_terra'] = match_satellite_insitu(ukl_modis_terra, ukl_insitu, MATCH_TOLERANCE_DAYS)
    print(f"\nMODIS-Terra matches: {len(matches['modis_terra'])}")
    if len(matches['modis_terra']) > 0:
        print(f"  Satellite chl range: {matches['modis_terra']['satellite_value'].min():.1f} - "
              f"{matches['modis_terra']['satellite_value'].max():.1f} µg/L")
        print(f"  In situ chl range: {matches['modis_terra']['insitu_chl'].min():.1f} - "
              f"{matches['modis_terra']['insitu_chl'].max():.1f} µg/L")

# Combine all matches for overview
all_matches = []
for sensor, match_df in matches.items():
    if len(match_df) > 0:
        match_df['sensor_type'] = sensor
        all_matches.append(match_df)

if all_matches:
    combined_matches = pd.concat(all_matches, ignore_index=True)
    
    print(f"\n" + "="*60)
    print(f"Total matched observations: {len(combined_matches)}")
    print(f"Average time difference: {combined_matches['days_diff'].mean():.1f} days")
    print(f"Maximum time difference: {combined_matches['days_diff'].max():.0f} days")
    print("="*60)

## Improved Calibration Model Development

In [ ]:
# Define calibration functions
def linear_model(x, a, b):
    """Linear model: y = a*x + b"""
    return a * x + b

def exponential_model(x, a, b, c):
    """Exponential model: y = a * exp(b*x) + c"""
    return a * np.exp(b * x) + c

def power_model(x, a, b, c):
    """Power model: y = a * x^b + c"""
    return a * np.power(np.maximum(x, 1e-6), b) + c

def log_model(x, a, b):
    """Logarithmic model: y = a * log(x) + b"""
    return a * np.log(np.maximum(x, 1e-6)) + b

def fit_calibration_model(x, y, model_type='linear', sensor_name='Unknown'):
    """
    Fit calibration model to matched data.
    Uses robust regression to handle outliers.
    """
    if len(x) < 3:
        return None
    
    # Remove any remaining invalid values
    valid_mask = (x > 0) & (y > 0) & np.isfinite(x) & np.isfinite(y)
    x = x[valid_mask]
    y = y[valid_mask]
    
    if len(x) < 3:
        return None
    
    results = {'sensor': sensor_name, 'model_type': model_type, 'n_points': len(x)}
    
    try:
        if model_type == 'linear':
            # Use Huber regression for robustness to outliers
            lr = HuberRegressor()
            lr.fit(x.reshape(-1, 1), y)
            y_pred = lr.predict(x.reshape(-1, 1))
            
            results['params'] = [lr.coef_[0], lr.intercept_]
            results['equation'] = f"Chl = {lr.coef_[0]:.3f} * x + {lr.intercept_:.3f}"
            
        elif model_type == 'log':
            # Logarithmic fit
            popt, _ = curve_fit(log_model, x, y, p0=[1, np.mean(y)])
            y_pred = log_model(x, *popt)
            
            results['params'] = popt
            results['equation'] = f"Chl = {popt[0]:.3f} * log(x) + {popt[1]:.3f}"
            
        elif model_type == 'exponential':
            # Exponential fit with better initial guess
            p0 = [np.mean(y), 0.01, np.min(y)]  # Better initial guess
            bounds = ([0, -np.inf, 0], [np.inf, np.inf, np.inf])  # Ensure positive coefficients
            popt, _ = curve_fit(exponential_model, x, y, p0=p0, bounds=bounds, maxfev=5000)
            y_pred = exponential_model(x, *popt)
            
            results['params'] = popt
            results['equation'] = f"Chl = {popt[0]:.3f} * exp({popt[1]:.3f} * x) + {popt[2]:.3f}"
            
        elif model_type == 'power':
            # Power fit with constraints
            p0 = [1, 1, 0]  # Initial guess
            bounds = ([0, 0, -np.inf], [np.inf, 5, np.inf])  # Reasonable bounds
            popt, _ = curve_fit(power_model, x, y, p0=p0, bounds=bounds, maxfev=5000)
            y_pred = power_model(x, *popt)
            
            results['params'] = popt
            results['equation'] = f"Chl = {popt[0]:.3f} * x^{popt[1]:.3f} + {popt[2]:.3f}"
        
        # Calculate statistics
        results['r2'] = r2_score(y, y_pred)
        results['rmse'] = np.sqrt(mean_squared_error(y, y_pred))
        results['mae'] = mean_absolute_error(y, y_pred)
        results['bias'] = np.mean(y_pred - y)
        
        # Calculate relative statistics
        results['mape'] = np.mean(np.abs((y - y_pred) / y)) * 100
        results['rrmse'] = results['rmse'] / np.mean(y) * 100
        
        results['success'] = True
        
    except Exception as e:
        print(f"Model fitting failed for {sensor_name} ({model_type}): {e}")
        results['success'] = False
    
    return results

# Fit calibration models for each sensor
print("\n" + "="*60)
print("Developing calibration models...")
print("="*60)

calibration_models = {}

for sensor_key, match_df in matches.items():
    if len(match_df) == 0:
        continue
    
    # Use converted chlorophyll values for calibration
    x = match_df['satellite_value'].values  # Already in chlorophyll units
    y = match_df['insitu_chl'].values
    
    print(f"\n{sensor_key.upper()} Calibration:")
    print(f"Number of matched points: {len(x)}")
    
    if len(x) >= 3:
        # Try different model types
        model_types = ['linear', 'log', 'power', 'exponential']
        sensor_models = {}
        
        for model_type in model_types:
            model = fit_calibration_model(x, y, model_type, sensor_key)
            if model and model['success']:
                sensor_models[model_type] = model
                print(f"  {model_type:12s}: R² = {model['r2']:.3f}, RMSE = {model['rmse']:.2f} µg/L")
        
        if sensor_models:
            # Select best model based on R²
            best_model_type = max(sensor_models.keys(), key=lambda k: sensor_models[k]['r2'])
            calibration_models[sensor_key] = sensor_models[best_model_type]
            print(f"  Best model: {best_model_type} (R² = {sensor_models[best_model_type]['r2']:.3f})")
            print(f"  Equation: {sensor_models[best_model_type]['equation']}")
    else:
        print(f"  Insufficient data points for calibration (need ≥3, have {len(x)})")

print(f"\n" + "="*60)
print(f"Calibration complete. Developed models for {len(calibration_models)} sensors.")
print("="*60)

## Enhanced Calibration Visualization

In [ ]:
# Create enhanced calibration plots
if calibration_models:
    n_sensors = len(calibration_models)
    fig, axes = plt.subplots(1, n_sensors, figsize=(7*n_sensors, 6))
    
    if n_sensors == 1:
        axes = [axes]
    
    for i, (sensor_key, model) in enumerate(calibration_models.items()):
        ax = axes[i]
        
        # Get data for this sensor
        match_df = matches[sensor_key]
        x = match_df['satellite_value'].values
        y = match_df['insitu_chl'].values
        
        # Scatter plot with color by time difference
        scatter = ax.scatter(x, y, alpha=0.6, s=50, 
                           c=match_df['days_diff'].values, 
                           cmap='viridis', label='Observations')
        plt.colorbar(scatter, ax=ax, label='Days difference')
        
        # Model line
        x_range = np.linspace(max(0.1, x.min()), x.max(), 100)
        
        if model['model_type'] == 'linear':
            y_model = linear_model(x_range, *model['params'])
        elif model['model_type'] == 'log':
            y_model = log_model(x_range, *model['params'])
        elif model['model_type'] == 'exponential':
            y_model = exponential_model(x_range, *model['params'])
        elif model['model_type'] == 'power':
            y_model = power_model(x_range, *model['params'])
        
        ax.plot(x_range, y_model, 'r-', linewidth=2, 
                label=f"{model['model_type'].title()} fit")
        
        # 1:1 line
        min_val = max(0, min(x.min(), y.min()))
        max_val = max(x.max(), y.max())
        ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='1:1 line')
        
        # Labels and title
        ax.set_xlabel('Satellite Chlorophyll (µg/L)', fontsize=11)
        ax.set_ylabel('In Situ Chlorophyll (µg/L)', fontsize=11)
        ax.set_title(f'{sensor_key.replace("_", " ").title()} Calibration\n'
                    f'R² = {model["r2"]:.3f}, RMSE = {model["rmse"]:.2f} µg/L\n'
                    f'n = {len(x)} points', fontsize=12)
        
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper left')
        
        # Add equation as text
        ax.text(0.05, 0.95, model['equation'], transform=ax.transAxes,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.9),
                verticalalignment='top', fontsize=10)
        
        # Add statistics text
        stats_text = f"MAE: {model['mae']:.2f} µg/L\nBias: {model['bias']:.2f} µg/L\nMAPE: {model['mape']:.1f}%"
        ax.text(0.95, 0.05, stats_text, transform=ax.transAxes,
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9),
                verticalalignment='bottom', horizontalalignment='right', fontsize=9)
    
    plt.suptitle('Satellite Chlorophyll Calibration Models', fontsize=14, y=1.02)
    plt.tight_layout()
    
    # Save calibration plot
    calibration_plot_path = OUTPUT_DIR / 'calibration_models.png'
    fig.savefig(calibration_plot_path, dpi=300, bbox_inches='tight')
    print(f"Calibration plot saved: {calibration_plot_path}")
    
    plt.show()
else:
    print("No calibration models to plot.")

## Apply Calibrations with Proper Filtering

In [ ]:
def apply_calibration(satellite_df, calibration_model):
    """
    Apply calibration model to satellite data.
    Properly handles NDCI to chlorophyll conversion.
    """
    if not calibration_model or not calibration_model['success']:
        return satellite_df.copy()
    
    df = satellite_df.copy()
    
    # Filter to positive values only
    df = df[df['value'] > 0].copy()
    
    # Convert NDCI to initial chlorophyll if needed
    if df['value_type'].iloc[0] == 'NDCI':
        df['initial_chl'] = convert_ndci_to_initial_chl(df['value'].values)
        x = df['initial_chl'].values
    else:
        df['initial_chl'] = df['value']
        x = df['value'].values
    
    # Apply calibration model
    if calibration_model['model_type'] == 'linear':
        df['calibrated_chl'] = linear_model(x, *calibration_model['params'])
    elif calibration_model['model_type'] == 'log':
        df['calibrated_chl'] = log_model(x, *calibration_model['params'])
    elif calibration_model['model_type'] == 'exponential':
        df['calibrated_chl'] = exponential_model(x, *calibration_model['params'])
    elif calibration_model['model_type'] == 'power':
        df['calibrated_chl'] = power_model(x, *calibration_model['params'])
    
    # Ensure reasonable range
    df['calibrated_chl'] = np.clip(df['calibrated_chl'], MIN_CHLOROPHYLL, MAX_CHLOROPHYLL)
    
    return df

# Apply calibrations to all datasets
print("="*60)
print("Applying calibrations to satellite data...")
print("="*60)

calibrated_data = {}

# Upper Klamath Lake
if 'sentinel' in calibration_models:
    calibrated_data['ukl_sentinel'] = apply_calibration(ukl_sentinel, calibration_models['sentinel'])
    print(f"\nApplied Sentinel-2 calibration to UKL data:")
    print(f"  Records: {len(calibrated_data['ukl_sentinel'])}")
    if len(calibrated_data['ukl_sentinel']) > 0:
        print(f"  Calibrated range: {calibrated_data['ukl_sentinel']['calibrated_chl'].min():.1f} - "
              f"{calibrated_data['ukl_sentinel']['calibrated_chl'].max():.1f} µg/L")
        print(f"  Mean: {calibrated_data['ukl_sentinel']['calibrated_chl'].mean():.1f} ± "
              f"{calibrated_data['ukl_sentinel']['calibrated_chl'].std():.1f} µg/L")

if 'modis_aqua' in calibration_models:
    calibrated_data['ukl_modis_aqua'] = apply_calibration(ukl_modis_aqua, calibration_models['modis_aqua'])
    print(f"\nApplied MODIS-Aqua calibration to UKL data:")
    print(f"  Records: {len(calibrated_data['ukl_modis_aqua'])}")
    if len(calibrated_data['ukl_modis_aqua']) > 0:
        print(f"  Calibrated range: {calibrated_data['ukl_modis_aqua']['calibrated_chl'].min():.1f} - "
              f"{calibrated_data['ukl_modis_aqua']['calibrated_chl'].max():.1f} µg/L")

if 'modis_terra' in calibration_models:
    calibrated_data['ukl_modis_terra'] = apply_calibration(ukl_modis_terra, calibration_models['modis_terra'])
    print(f"\nApplied MODIS-Terra calibration to UKL data:")
    print(f"  Records: {len(calibrated_data['ukl_modis_terra'])}")
    if len(calibrated_data['ukl_modis_terra']) > 0:
        print(f"  Calibrated range: {calibrated_data['ukl_modis_terra']['calibrated_chl'].min():.1f} - "
              f"{calibrated_data['ukl_modis_terra']['calibrated_chl'].max():.1f} µg/L")

# Detroit Lake
print("\n" + "-"*40)
if 'sentinel' in calibration_models:
    calibrated_data['detroit_sentinel'] = apply_calibration(detroit_sentinel, calibration_models['sentinel'])
    print(f"\nApplied Sentinel-2 calibration to Detroit data:")
    print(f"  Records: {len(calibrated_data['detroit_sentinel'])}")
    if len(calibrated_data['detroit_sentinel']) > 0:
        print(f"  Calibrated range: {calibrated_data['detroit_sentinel']['calibrated_chl'].min():.1f} - "
              f"{calibrated_data['detroit_sentinel']['calibrated_chl'].max():.1f} µg/L")
        print(f"  Mean: {calibrated_data['detroit_sentinel']['calibrated_chl'].mean():.1f} ± "
              f"{calibrated_data['detroit_sentinel']['calibrated_chl'].std():.1f} µg/L")

if 'modis_aqua' in calibration_models:
    calibrated_data['detroit_modis_aqua'] = apply_calibration(detroit_modis_aqua, calibration_models['modis_aqua'])
    print(f"\nApplied MODIS-Aqua calibration to Detroit data:")
    print(f"  Records: {len(calibrated_data['detroit_modis_aqua'])}")
    if len(calibrated_data['detroit_modis_aqua']) > 0:
        print(f"  Calibrated range: {calibrated_data['detroit_modis_aqua']['calibrated_chl'].min():.1f} - "
              f"{calibrated_data['detroit_modis_aqua']['calibrated_chl'].max():.1f} µg/L")

if 'modis_terra' in calibration_models:
    calibrated_data['detroit_modis_terra'] = apply_calibration(detroit_modis_terra, calibration_models['modis_terra'])
    print(f"\nApplied MODIS-Terra calibration to Detroit data:")
    print(f"  Records: {len(calibrated_data['detroit_modis_terra'])}")
    if len(calibrated_data['detroit_modis_terra']) > 0:
        print(f"  Calibrated range: {calibrated_data['detroit_modis_terra']['calibrated_chl'].min():.1f} - "
              f"{calibrated_data['detroit_modis_terra']['calibrated_chl'].max():.1f} µg/L")

print(f"\n" + "="*60)
print(f"Calibration applied to {len(calibrated_data)} datasets.")
print("="*60)

## Comprehensive Time Series Comparison

In [ ]:
# Create comprehensive comparison plots
def create_comparison_plot(raw_data, calibrated_data, title, save_name, include_insitu=None):
    """
    Create time series plot comparing raw and calibrated data.
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
    
    colors = {'sentinel': 'blue', 'modis_aqua': 'green', 'modis_terra': 'red'}
    
    # Raw data subplot
    for sensor_type, df in raw_data.items():
        if len(df) > 0:
            sensor_key = sensor_type.split('_', 1)[1] if '_' in sensor_type else sensor_type
            color = colors.get(sensor_key, 'gray')
            
            # For NDCI, convert to initial chlorophyll
            if df['value_type'].iloc[0] == 'NDCI':
                y_values = convert_ndci_to_initial_chl(df['value'].values)
                label = f"{sensor_key.replace('_', '-').title()} (NDCI→Chl)"
            else:
                y_values = df['value']
                label = f"{sensor_key.replace('_', '-').title()} (Raw)"
            
            ax1.scatter(df['date'], y_values, color=color, alpha=0.3, s=10, label=label)
    
    ax1.set_ylabel('Chlorophyll-a (µg/L)')
    ax1.set_title(f'{title} - Raw/Initial Estimates')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper left')
    ax1.set_ylim(bottom=0)
    
    # Calibrated data subplot
    for sensor_type, df in calibrated_data.items():
        if len(df) > 0 and 'calibrated_chl' in df.columns:
            sensor_key = sensor_type.split('_', 1)[1] if '_' in sensor_type else sensor_type
            color = colors.get(sensor_key, 'gray')
            label = f"{sensor_key.replace('_', '-').title()} (Calibrated)"
            
            ax2.scatter(df['date'], df['calibrated_chl'], color=color, alpha=0.5, s=15, label=label)
    
    # Add in situ data if provided
    if include_insitu is not None and len(include_insitu) > 0:
        ax2.scatter(include_insitu['date'], include_insitu['chlorophyll_ugL'],
                   color='black', s=30, marker='o', label='In Situ', alpha=0.8, zorder=10)
    
    ax2.set_xlabel('Date')
    ax2.set_ylabel('Chlorophyll-a (µg/L)')
    ax2.set_title(f'{title} - Calibrated')
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc='upper left')
    ax2.set_ylim(bottom=0)
    
    plt.tight_layout()
    
    # Save plot
    plot_path = OUTPUT_DIR / f'{save_name}.png'
    fig.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"Comparison plot saved: {plot_path}")
    
    return fig

# Upper Klamath Lake comparison
ukl_raw = {'ukl_sentinel': ukl_sentinel, 'ukl_modis_aqua': ukl_modis_aqua, 'ukl_modis_terra': ukl_modis_terra}
ukl_cal = {k: v for k, v in calibrated_data.items() if 'ukl' in k}

if ukl_cal:
    fig_ukl = create_comparison_plot(
        ukl_raw, ukl_cal,
        'Upper Klamath Lake',
        'UKL_comparison',
        include_insitu=ukl_insitu
    )
    plt.show()

# Detroit Lake comparison
detroit_raw = {'detroit_sentinel': detroit_sentinel, 'detroit_modis_aqua': detroit_modis_aqua, 
               'detroit_modis_terra': detroit_modis_terra}
detroit_cal = {k: v for k, v in calibrated_data.items() if 'detroit' in k}

if detroit_cal:
    fig_detroit = create_comparison_plot(
        detroit_raw, detroit_cal,
        'Detroit Lake',
        'Detroit_comparison'
    )
    plt.show()

## Export Calibrated Data and Summary

In [ ]:
# Export calibrated datasets
print("="*60)
print("Exporting calibrated datasets...")
print("="*60)

export_dir = OUTPUT_DIR / 'calibrated_data'
export_dir.mkdir(exist_ok=True)

for dataset_name, df in calibrated_data.items():
    if len(df) > 0 and 'calibrated_chl' in df.columns:
        # Prepare export dataframe
        export_df = df[['date', 'value', 'calibrated_chl', 'sensor']].copy()
        
        # Add initial chlorophyll if it exists
        if 'initial_chl' in df.columns:
            export_df['initial_chl'] = df['initial_chl']
        
        export_df.columns = ['date', 'raw_value'] + list(export_df.columns[2:])
        
        # Sort by date
        export_df = export_df.sort_values('date')
        
        # Export to CSV
        export_path = export_dir / f'{dataset_name}_calibrated.csv'
        export_df.to_csv(export_path, index=False)
        print(f"Exported {dataset_name}: {len(export_df)} records")

# Export calibration summary
if calibration_models:
    summary = []
    for sensor, model in calibration_models.items():
        summary.append({
            'sensor': sensor,
            'model_type': model['model_type'],
            'equation': model['equation'],
            'r2': model['r2'],
            'rmse': model['rmse'],
            'n_points': model['n_points']
        })
    
    summary_df = pd.DataFrame(summary)
    summary_path = OUTPUT_DIR / 'calibration_summary.csv'
    summary_df.to_csv(summary_path, index=False)
    print(f"\nCalibration summary saved: {summary_path}")
    
    print("\n" + "="*60)
    print("CALIBRATION SUMMARY")
    print("="*60)
    print(summary_df.to_string(index=False))

print("\n" + "="*60)
print("ANALYSIS COMPLETE")
print(f"All results saved to: {OUTPUT_DIR}")
print("="*60)